# Lunar Tear Patcher (iOS)

Patch a NieR Re\[in\]carnation IPA and master data for use with a private server.

**How to use:**
1. Fill in the **Configuration** form below.
2. Run cells top-to-bottom, or use **Runtime → Run all**.
3. Patched files will download to your browser.
4. Re-sign the IPA locally before installing — Colab cannot sign for you (no Apple developer cert). See the **Re-sign** cell below for options.

**Two modes:**
- **Drive mode** (default): Check **use_gdrive**, enter paths relative to `/content/drive/` (e.g. `MyDrive/original.ipa`).
- **URL mode**: Uncheck **use_gdrive**, paste download URLs into `ipa_source` / `masterdata_source`.

Each section is independent — skip what you don't need.

In [ ]:
#@title Configuration

#@markdown ### Source Mode
use_gdrive = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### File Sources
#@markdown With **use_gdrive** checked, enter Drive paths relative to `/content/drive/`
#@markdown (e.g. `MyDrive/original.ipa`). Otherwise, enter download URLs.
ipa_source = "" #@param {type:"string"}
masterdata_source = "" #@param {type:"string"}

#@markdown ---
#@markdown ### Server Settings
grpc_addr = "10.0.2.2:443" #@param {type:"string"}
http_addr = "10.0.2.2:8080" #@param {type:"string"}
auth_host = "" #@param {type:"string"}
scripts_branch = "main" #@param {type:"string"}

import os

if use_gdrive:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted at /content/drive\n')
    for label, src in [('IPA', ipa_source), ('Master data', masterdata_source)]:
        if not src:
            print(f'  {label}: (not set, will skip)')
            continue
        path = os.path.join('/content/drive', src)
        if os.path.isfile(path):
            size = os.path.getsize(path)
            print(f'  {label}: {path} — OK ({size / 1024:.1f} KB)')
        else:
            print(f'  {label}: {path} — NOT FOUND')
else:
    print('URL mode — files will be downloaded during patching.')

In [ ]:
#@title Install dependencies
import os

if not os.path.isdir('/content/scripts'):
    !git clone --depth 1 --branch {scripts_branch} https://gitlab.com/walter-sparrow-group/lunar-scripts.git /content/scripts
else:
    !cd /content/scripts && git fetch origin {scripts_branch} && git checkout {scripts_branch} && git pull origin {scripts_branch}
    print(f'Scripts updated to branch: {scripts_branch}')

!pip install -q pycryptodome msgpack lz4

print('\nDone.')

In [ ]:
#@title Patch IPA
import urllib.request, shutil, os
from google.colab import files

assert ipa_source, 'Set ipa_source in the Configuration cell first.'
assert grpc_addr, 'Set grpc_addr in the Configuration cell first.'
assert http_addr, 'Set http_addr in the Configuration cell first.'

if use_gdrive:
    ipa_path = os.path.join('/content/drive', ipa_source)
    assert os.path.isfile(ipa_path), f'File not found: {ipa_path}'
    shutil.copy2(ipa_path, '/content/original.ipa')
    size_mb = os.path.getsize('/content/original.ipa') / 1024 / 1024
    print(f'Copied IPA from Drive: {ipa_path} ({size_mb:.1f} MB)')
else:
    print(f'Downloading IPA from {ipa_source} ...')
    urllib.request.urlretrieve(ipa_source, '/content/original.ipa')
    size_mb = os.path.getsize('/content/original.ipa') / 1024 / 1024
    print(f'  Downloaded {size_mb:.1f} MB')

auth_flag = f'--auth-host {auth_host}' if auth_host else ''
print('\nPatching ...')
!python /content/scripts/ios/patch_ipa.py /content/original.ipa --grpc-addr {grpc_addr} --http-addr {http_addr} {auth_flag} -o /content/patched.ipa

size_mb = os.path.getsize('/content/patched.ipa') / 1024 / 1024
print(f'\nDone! Patched IPA: {size_mb:.1f} MB')

files.download('/content/patched.ipa')

## Re-sign the patched IPA

The patched IPA is **not signed**. iOS won't install it as-is. Pick whichever option matches your setup:

- **AltStore / Sideloadly**: import `patched.ipa` — auto-resigns with your free or paid Apple ID and installs over USB / Wi-Fi.
- **Jailbroken device**: `ldid -S patched.ipa` then install with Filza / TrollStore / etc.
- **macOS with a developer cert**:
  ```
  unzip patched.ipa -d patched_dir
  codesign -f -s 'iPhone Developer: NAME' patched_dir/Payload/NieR.app
  cd patched_dir && zip -r ../patched-signed.ipa Payload
  ```

If you do not have any of the above, install AltStore on macOS/Windows first; it is the simplest path.

In [ ]:
#@title Patch Master Data
import urllib.request, shutil, os
from google.colab import files

assert masterdata_source, 'Set masterdata_source in the Configuration cell first.'

if use_gdrive:
    md_path = os.path.join('/content/drive', masterdata_source)
    assert os.path.isfile(md_path), f'File not found: {md_path}'
    shutil.copy2(md_path, '/content/masterdata.bin.e')
    size_kb = os.path.getsize('/content/masterdata.bin.e') / 1024
    print(f'Copied master data from Drive: {md_path} ({size_kb:.1f} KB)')
else:
    print(f'Downloading master data from {masterdata_source} ...')
    urllib.request.urlretrieve(masterdata_source, '/content/masterdata.bin.e')
    size_kb = os.path.getsize('/content/masterdata.bin.e') / 1024
    print(f'  Downloaded {size_kb:.1f} KB')

print('\nPatching ...')
!python /content/scripts/patch_masterdata.py \
  --input /content/masterdata.bin.e \
  --output /content/20240404193219.bin.e

size_kb = os.path.getsize('/content/20240404193219.bin.e') / 1024
print(f'\nDone! Patched master data: {size_kb:.1f} KB')

files.download('/content/20240404193219.bin.e')